In [ ]:
# TO RUN ON COLAB ONLY

# installs
!pip install tweet-preprocessor==0.5.0 feedparser whoosh iterative-stratification fastapi uvicorn
!pip install nltk spacy

# imports
import sys
from pathlib import Path
from google.colab import drive
import pandas as pd
import numpy as np
import torch

# mount google drive and set path-related variables
drive.mount('/content/drive')
BASE_DIR   = Path('/content/drive/MyDrive/linguistic_markers')
SPRINT_DIR = BASE_DIR / '581_Sprint_3'
SRC_DIR    = SPRINT_DIR / 'src'
DATA_DIR   = BASE_DIR / 'data' / 'final_splits'
SAVE_DIR   = SPRINT_DIR / 'saved_models'

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(1, str(SPRINT_DIR))
sys.path.insert(2, str(SRC_DIR))

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

from config import FASTTEXT_PATH, TARGETS, SEED
from preprocess import preprocess
from metrics import compute_metrics
import cnn_baseline as cnn
import cnn_mtl_ling
import cnn_mtl_no_ling as cnn_mtl_pos
import joblib, pickle
import torch.nn as nn

import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

In [ ]:
# Model name constants — must match TrainAndSaveModels.ipynb exactly
M_TEXT_CNN                 = "TextCNN"
M_TEXT_CNN_TRANSFER        = "TextCNN Transfer"
M_TEXT_CNN_TRANSFER_SHARED = "TextCNN_Transfer_shared"
M_CNN_MTL_LING             = "TextCNN MTL (POS + linguistic)"
M_CNN_MTL_POS              = "TextCNN MTL (POS)"
M_LOGREG                   = "LogReg"
M_LOGREG_EMBED             = "LogReg (+ embeddings)"
M_LOGREG_MTL_POS           = "LogReg MTL (cascaded POS)"

M_ENSEMBLE_SOFT_VOTE       = "Soft Vote Ensemble"
M_ENSEMBLE_MOTIVATED       = "Motivated Ensemble"
M_ENSEMBLE_LEAN_SOFT_VOTE  = "Lean Soft Vote"
M_ENSEMBLE_LEAN_MOTIVATED  = "Lean Motivated Ensemble"

CNN_MODELS      = [M_TEXT_CNN, M_TEXT_CNN_TRANSFER, M_CNN_MTL_LING, M_CNN_MTL_POS]
LR_MODELS       = [M_LOGREG, M_LOGREG_EMBED, M_LOGREG_MTL_POS]
BASE_MODELS     = LR_MODELS + CNN_MODELS
ENSEMBLE_MODELS = [M_ENSEMBLE_SOFT_VOTE, M_ENSEMBLE_MOTIVATED,
                   M_ENSEMBLE_LEAN_SOFT_VOTE, M_ENSEMBLE_LEAN_MOTIVATED]

In [ ]:
# Rebuild vocab and embedding matrix — required to instantiate CNN models
train_rows = cnn.load_csv(DATA_DIR / 'mis_df_train.csv')
dev_rows   = cnn.load_csv(DATA_DIR / 'mis_df_dev.csv')
vocab        = cnn.build_vocab(train_rows, preprocess)
embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
print(f'Vocab size: {len(vocab)} | Embed matrix: {embed_matrix.shape}')

In [ ]:
# Data loader helpers — call whichever matches the model you want to run inference with

def get_loaders_baseline():
    """Standard loaders for TextCNN / TextCNN Transfer."""
    train = cnn.make_loader(train_rows, vocab, shuffle=False, tokenize_fn=preprocess)
    dev   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)
    return train, dev

def get_loaders_ling():
    """Loaders for TextCNN MTL (POS + linguistic)."""
    train = cnn_mtl_ling.make_loader(train_rows, vocab, shuffle=False, tokenize_fn=preprocess)
    dev   = cnn_mtl_ling.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)
    return train, dev

def get_loaders_pos():
    """Loaders for TextCNN MTL (POS only)."""
    train = cnn_mtl_pos.make_loader(train_rows, vocab, shuffle=False, tokenize_fn=preprocess)
    dev   = cnn_mtl_pos.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)
    return train, dev

In [ ]:
# Maps each CNN model constant to the module that owns its TextCNN class
_CNN_CLASS = {
    M_TEXT_CNN:                 cnn.TextCNN,
    M_TEXT_CNN_TRANSFER:        cnn.TextCNN,
    M_TEXT_CNN_TRANSFER_SHARED: cnn.TextCNN,
    M_CNN_MTL_LING:             cnn_mtl_ling.TextCNN,
    M_CNN_MTL_POS:              cnn_mtl_pos.TextCNN,
}

def _safe_key(key):
    return (key.replace(' ', '_').replace('/', '-')
               .replace('(', '').replace(')', '')
               .replace('+', 'plus'))


def load_cnn_model(model_name, target=None):
    """
    Load a saved TextCNN state dict and return the model in eval mode.

    Pass the model constant (e.g. M_TEXT_CNN) and optionally a target string.
    If target is provided, the full key is f"{model_name} — {target}".
    For the shared transfer weights use M_TEXT_CNN_TRANSFER_SHARED (no target needed).
    """
    key = f"{model_name} — {target}" if target else model_name
    path = SAVE_DIR / f"{_safe_key(key)}.pt"
    if not path.exists():
        raise FileNotFoundError(f'No saved model at {path}')

    model_cls = _CNN_CLASS.get(model_name)
    if model_cls is None:
        raise ValueError(f'Unknown CNN model name: {model_name!r}. Expected one of: {list(_CNN_CLASS)}')

    model = model_cls(len(vocab), embed_matrix).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    print(f'Loaded: {path.name}')
    return model


def load_result(model_name, target=None):
    """
    Load a saved (preds, labels, probs) result tuple.

    Pass the model constant and optionally a target string.
    Works for base models, LogReg variants, and all ensembles.
    """
    key = f"{model_name} — {target}" if target else model_name
    path = SAVE_DIR / f"{_safe_key(key)}_result.pkl"
    if not path.exists():
        raise FileNotFoundError(f'No saved result at {path}')
    with open(path, 'rb') as f:
        result = pickle.load(f)
    print(f'Loaded: {path.name}')
    return result


def list_saved():
    """Print all files in SAVE_DIR grouped by type."""
    pts      = sorted(SAVE_DIR.glob('*.pt'))
    joblibfs = sorted(SAVE_DIR.glob('*.joblib'))
    pkls     = sorted(SAVE_DIR.glob('*.pkl'))
    print(f'PyTorch state dicts ({len(pts)}):')
    for p in pts:      print(f'  {p.name}')
    print(f'\nsklearn models ({len(joblibfs)}):')
    for p in joblibfs: print(f'  {p.name}')
    print(f'\nResult tuples ({len(pkls)}):')
    for p in pkls:     print(f'  {p.name}')

In [ ]:
# Example usage — swap in any constant to load the model you want
list_saved()

# Load a CNN model and run inference on dev set
# _, dev_loader = get_loaders_baseline()
# model = load_cnn_model(M_TEXT_CNN, target='misinformation_label')
# preds, labels, probs = cnn.predict(model, dev_loader, DEVICE, target='misinformation_label')
# print(compute_metrics(preds, labels, probs))

# Load the shared transfer weights (no target suffix)
# model = load_cnn_model(M_TEXT_CNN_TRANSFER_SHARED)

# Load a base model result tuple 
# preds, labels, probs = load_result(M_TEXT_CNN, target='opinion_label')

# Load an ensemble result tuple
# preds, labels, probs = load_result(M_ENSEMBLE_LEAN_MOTIVATED, target='misinformation_label')

### Load All Models

In [ ]:
# CNN models with per-target state dicts (Transfer is excluded — it only has shared weights and didn't perform well anyway)
_CNN_MODELS_PER_TARGET = [M_TEXT_CNN, M_CNN_MTL_LING, M_CNN_MTL_POS]

# Load all CNN state dicts 
models = {}

models[M_TEXT_CNN_TRANSFER_SHARED] = load_cnn_model(M_TEXT_CNN_TRANSFER_SHARED)

for model_name in _CNN_MODELS_PER_TARGET:
    for target in TARGETS:
        key = f"{model_name} — {target}"
        models[key] = load_cnn_model(model_name, target=target)

print(f"\nLoaded {len(models)} CNN model(s).")

# Load all result tuples (base models + ensembles)
results = {}

for model_name in BASE_MODELS + ENSEMBLE_MODELS:
    for target in TARGETS:
        key = f"{model_name} — {target}"
        results[key] = load_result(model_name, target=target)

print(f"\nLoaded {len(results)} result tuple(s).")